# Data Manipulation with Pandas — Week 22 Assignment

**Name:** Isaac Ogbaji  
**Course:** Data Analysis  
**Topic:** Data Manipulation with Pandas  
**Notebook Name:** `Pandas_Week22_IsaacOgbaji.ipynb`

## Assignment Objectives

This notebook demonstrates how to:

1. Load data into Pandas DataFrames from different formats: **CSV, JSON, Excel, and TXT**.
2. Clean datasets by handling missing values, duplicates, inconsistent column names, and incorrect data types.
3. Apply filtering and grouping operations to summarize data.
4. Save cleaned datasets for future analysis.

> Note: The TXT dataset was not uploaded because of the upload limit. To still demonstrate TXT loading, this notebook creates a simple TXT file from the CSV dataset and then loads it back into Pandas.

## 1. Import Required Libraries

Pandas is used for data loading, cleaning, filtering, grouping, and exploratory analysis. NumPy is used for missing values and numerical operations.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. File Names

The uploaded datasets are saved in the same folder as this notebook.

In [ ]:
csv_file = "countries of the world.csv"
json_file = "json_sample.json"
excel_file = "world_population_excel_workbook.xlsx"
txt_file = "countries_sample.txt"

for file in [csv_file, json_file, excel_file]:
    print(f"{file}: {'Found' if os.path.exists(file) else 'Not found'}")

## 3. Load Data from CSV

The CSV file contains country names and their regions.

In [ ]:
countries_df = pd.read_csv(csv_file)
print("CSV Data Loaded Successfully")
print("Shape:", countries_df.shape)
countries_df.head()

## 4. Load Data from Excel

The Excel workbook contains world population data. We load the first sheet because it contains the full population dataset.

In [ ]:
excel_sheets = pd.ExcelFile(excel_file).sheet_names
print("Available Excel Sheets:", excel_sheets)

population_df = pd.read_excel(excel_file, sheet_name=0, engine="openpyxl")
print("Excel Data Loaded Successfully")
print("Shape:", population_df.shape)
population_df.head()

## 5. Load Data from JSON

The JSON data contains movie information. Because the movie titles are stored as keys, we convert the JSON object into a DataFrame using `pd.DataFrame.from_dict()`.

In [ ]:
with open(json_file, "r", encoding="utf-8") as file:
    raw_json = json.load(file)

# The JSON file is a list containing one dictionary of movies.
movie_dictionary = raw_json[0]

movies_df = (
    pd.DataFrame.from_dict(movie_dictionary, orient="index")
    .reset_index()
    .rename(columns={"index": "Movie"})
)

print("JSON Data Loaded Successfully")
print("Shape:", movies_df.shape)
movies_df.head()

## 6. Create and Load a TXT Dataset

Since the TXT file was not uploaded, we create a tab-separated TXT file from the CSV data, then load it with Pandas.

In [ ]:
countries_df.to_csv(txt_file, sep="	", index=False)

txt_df = pd.read_csv(txt_file, sep="	")
print("TXT Data Loaded Successfully")
print("Shape:", txt_df.shape)
txt_df.head()

## 7. Helper Function for Cleaning Column Names

This function makes column names easier to work with by converting them to lowercase and replacing spaces with underscores.

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("²", "2", regex=False)
        .str.replace("â²", "2", regex=False)
        .str.replace("Â²", "2", regex=False)
        .str.replace("â", "", regex=False)
    )
    return df

## 8. Clean the CSV Countries Dataset

Cleaning steps:

- Standardize column names.
- Remove extra spaces from text values.
- Check and remove duplicates.
- Handle missing values.

In [ ]:
countries_clean = clean_column_names(countries_df)

# Remove extra spaces from text columns
for column in countries_clean.select_dtypes(include="object").columns:
    countries_clean[column] = countries_clean[column].astype(str).str.strip()

print("Missing values before cleaning:")
print(countries_clean.isna().sum())

print("\nDuplicate rows before cleaning:", countries_clean.duplicated().sum())

countries_clean = countries_clean.drop_duplicates()
countries_clean["region"] = countries_clean["region"].replace("nan", np.nan).fillna("Unknown")

print("\nMissing values after cleaning:")
print(countries_clean.isna().sum())

print("\nCleaned CSV shape:", countries_clean.shape)
countries_clean.head()

## 9. Clean the Excel Population Dataset

Cleaning steps:

- Standardize column names.
- Remove duplicates.
- Strip spaces from text values.
- Convert population and area columns to numeric data types.
- Fill missing numeric values with the median.
- Fill missing text values with `Unknown`.

In [ ]:
population_clean = clean_column_names(population_df)

for column in population_clean.select_dtypes(include="object").columns:
    population_clean[column] = population_clean[column].astype(str).str.strip()

print("Missing values before cleaning:")
print(population_clean.isna().sum())

print("\nDuplicate rows before cleaning:", population_clean.duplicated().sum())

population_clean = population_clean.drop_duplicates()

# Convert likely numeric columns to numeric
for column in population_clean.columns:
    if any(keyword in column for keyword in ["population", "area", "density", "growth", "percentage", "rank"]):
        population_clean[column] = pd.to_numeric(population_clean[column], errors="coerce")

# Fill missing values
numeric_columns = population_clean.select_dtypes(include=np.number).columns
text_columns = population_clean.select_dtypes(include="object").columns

population_clean[numeric_columns] = population_clean[numeric_columns].fillna(population_clean[numeric_columns].median())
population_clean[text_columns] = population_clean[text_columns].replace("nan", np.nan).fillna("Unknown")

print("\nMissing values after cleaning:")
print(population_clean.isna().sum())

print("\nCleaned Excel shape:", population_clean.shape)
population_clean.head()

## 10. Clean the JSON Movies Dataset

The JSON file has inconsistent column names, for example `Popcorn Score` and `popcornscore`. We clean and merge these columns into standard names.

Cleaning steps:

- Standardize column names.
- Merge duplicate meaning columns.
- Replace values like `unknown`, `unkown`, and `-1` with missing values.
- Convert score and gross columns to numeric.
- Fill missing values appropriately.

In [ ]:
movies_clean = clean_column_names(movies_df)

# Combine duplicate columns created after standardizing names, such as Rating/rating.
movies_clean = movies_clean.T.groupby(level=0).first().T

# Merge columns that mean the same thing but have different names
if "popcornscore" in movies_clean.columns and "popcorn_score" in movies_clean.columns:
    movies_clean["popcorn_score"] = movies_clean["popcorn_score"].combine_first(movies_clean["popcornscore"])
    movies_clean = movies_clean.drop(columns=["popcornscore"])

if "tomatoscore" in movies_clean.columns and "tomato_score" in movies_clean.columns:
    movies_clean["tomato_score"] = movies_clean["tomato_score"].combine_first(movies_clean["tomatoscore"])
    movies_clean = movies_clean.drop(columns=["tomatoscore"])

if "rating" in movies_clean.columns and "rating" not in movies_clean.columns:
    movies_clean["rating"] = movies_clean["rating"]

# Standardize text values
for column in movies_clean.select_dtypes(include="object").columns:
    movies_clean[column] = movies_clean[column].astype(str).str.strip()

# Replace known invalid/missing indicators
movies_clean = movies_clean.replace(["unknown", "unkown", "Unknown", -1, "-1", "nan"], np.nan)

# Clean gross column if present
if "gross" in movies_clean.columns:
    movies_clean["gross"] = (
        movies_clean["gross"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    movies_clean["gross"] = pd.to_numeric(movies_clean["gross"], errors="coerce")

# Convert score columns to numeric
for column in ["imdb_metascore", "popcorn_score", "tomato_score"]:
    if column in movies_clean.columns:
        movies_clean[column] = pd.to_numeric(movies_clean[column], errors="coerce")

print("Missing values before final filling:")
print(movies_clean.isna().sum())

movies_clean = movies_clean.drop_duplicates()

# Fill text columns
if "genre" in movies_clean.columns:
    movies_clean["genre"] = movies_clean["genre"].fillna("Unknown")
if "rating" in movies_clean.columns:
    movies_clean["rating"] = movies_clean["rating"].fillna("Unrated")

# Fill numeric columns with median
numeric_columns = movies_clean.select_dtypes(include=np.number).columns
movies_clean[numeric_columns] = movies_clean[numeric_columns].fillna(movies_clean[numeric_columns].median())

print("\nMissing values after cleaning:")
print(movies_clean.isna().sum())

print("\nCleaned JSON shape:", movies_clean.shape)
movies_clean.head()

## 11. Filtering Operations

Filtering helps us select rows that meet specific conditions.

In [ ]:
# Countries in Sub-Saharan Africa from the CSV dataset
sub_saharan_countries = countries_clean[countries_clean["region"].str.contains("SUB-SAHARAN AFRICA", case=False, na=False)]
print("Number of Sub-Saharan African countries:", len(sub_saharan_countries))
sub_saharan_countries.head(10)

In [ ]:
# Countries with 2022 population greater than 100 million
large_population = population_clean[population_clean["2022_population"] > 100_000_000]
large_population[["country", "continent", "2022_population", "density_per_km2"]].sort_values("2022_population", ascending=False)

In [ ]:
# Movies with tomato score greater than or equal to 80
high_tomato_score = movies_clean[movies_clean["tomato_score"] >= 80]
high_tomato_score[["movie", "rating", "tomato_score", "popcorn_score"]].sort_values("tomato_score", ascending=False).head(10)

## 12. Grouping Operations

Grouping allows us to summarize data by categories.

In [ ]:
# Count countries by region from CSV data
countries_by_region = (
    countries_clean
    .groupby("region")
    .size()
    .reset_index(name="number_of_countries")
    .sort_values("number_of_countries", ascending=False)
)

countries_by_region

In [ ]:
# Population summary by continent from Excel data
population_by_continent = (
    population_clean
    .groupby("continent")
    .agg(
        total_2022_population=("2022_population", "sum"),
        average_2022_population=("2022_population", "mean"),
        average_density=("density_per_km2", "mean"),
        number_of_countries=("country", "count")
    )
    .reset_index()
    .sort_values("total_2022_population", ascending=False)
)

population_by_continent

In [ ]:
# Movie score summary by rating from JSON data
movie_summary_by_rating = (
    movies_clean
    .groupby("rating")
    .agg(
        number_of_movies=("movie", "count"),
        average_tomato_score=("tomato_score", "mean"),
        average_popcorn_score=("popcorn_score", "mean"),
        average_gross=("gross", "mean")
    )
    .reset_index()
    .sort_values("number_of_movies", ascending=False)
)

movie_summary_by_rating

## 13. Exploratory Data Analysis Summary

This section produces quick insights from the cleaned datasets.

In [ ]:
print("CSV Countries Dataset")
print("Total countries:", countries_clean["country"].nunique())
print("Total regions:", countries_clean["region"].nunique())

print("\nExcel Population Dataset")
print("Total countries:", population_clean["country"].nunique())
print("Total 2022 world population in dataset:", f"{population_clean['2022_population'].sum():,.0f}")
print("Most populated country:", population_clean.loc[population_clean["2022_population"].idxmax(), "country"])

print("\nJSON Movies Dataset")
print("Total movies:", movies_clean["movie"].nunique())
print("Average tomato score:", round(movies_clean["tomato_score"].mean(), 2))
print("Average popcorn score:", round(movies_clean["popcorn_score"].mean(), 2))

## 14. Save Cleaned Datasets

The cleaned datasets are saved as CSV files. These files can be uploaded to GitHub together with the notebook if needed.

In [ ]:
countries_clean.to_csv("cleaned_countries.csv", index=False)
population_clean.to_csv("cleaned_population.csv", index=False)
movies_clean.to_csv("cleaned_movies.csv", index=False)

print("Cleaned files saved successfully:")
print("- cleaned_countries.csv")
print("- cleaned_population.csv")
print("- cleaned_movies.csv")

## 15. Conclusion

In this assignment, I loaded data from **CSV, JSON, Excel, and TXT** formats into Pandas DataFrames. I cleaned the datasets by removing duplicates, handling missing values, standardizing column names, and converting data types. I also applied filtering and grouping operations to summarize country, population, and movie data.